Imports

In [1]:
from collections import deque, namedtuple
import gymnasium as gym 
import random
import torch 
import torch.nn as nn 
import torch.optim as optim 
import torch.nn.functional as F
from tqdm import tqdm 
import numpy as np  

In [2]:
class lunar_agent:
    def __init__(
            self,
            gamma,
            num_steps_for_update,
            learning_rate,
            epsilon,
            env = gym.make('LunarLander-v3')
            ):
        
        self.gamma = gamma 
        self.lr = learning_rate
        self.num_steps_for_update = num_steps_for_update 
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.env = env
        self.n_actions  = env.action_space.n 
        self.n_states = env.observation_space.shape[0]
        self.experience = namedtuple("Experience", field_names=["state", "action" , "reward" , "next_state", "done"])
        self.tau = 0.001
        self.epsilon = epsilon
        self.memory_size = 100000

    def create_nn(self):
        class QNetwork(nn.Module):
            def __init__ (self,n_states,n_actions):
                super (QNetwork,self).__init__()

                self.fc1 = nn.Linear(n_states,128)
                self.fc2 = nn.Linear(128,256)
                self.fc3 = nn.Linear(256,64)
                self.fc4 = nn.Linear(64,64)
                self.fc5 = nn.Linear(64, n_actions)
            
            def forward(self , state):
                x = F.relu(self.fc1(state))
                x = F.relu(self.fc2(x))
                x = F.relu(self.fc3(x))
                x = F.relu(self.fc4(x))
                return self.fc5(x)
            
        self.q_network = QNetwork(self.n_states,self.n_actions).to(self.device)
        self.target_q_network = QNetwork(self.n_states,self.n_actions).to(self.device)

        #sync the weights 

        self.target_q_network.load_state_dict(self.q_network.state_dict())
        self.optimizer = optim.Adam(self.q_network.parameters(), lr=self.lr)

    def compute_loss(self,experiences):


        states , actions, rewards, next_states , done_vals = experiences
        states = states.to(self.device)
        actions = actions.to(self.device)
        rewards = rewards.to(self.device)
        next_states = next_states.to(self.device)
        done_vals = done_vals.to(self.device)

        with torch.no_grad():
            max_q_next = self.target_q_network(next_states).max(1)[0]
            y_targets = rewards + self.gamma*max_q_next*(1-done_vals)

        q_expected = self.q_network(states).gather(1,actions.long().unsqueeze(1)).squeeze(1)

        loss = F.mse_loss(y_targets,q_expected)
        return loss
    
    def soft_update(self ):
        tau = self.tau
        
        for target_param, local_param in zip (self.target_q_network.parameters(), self.q_network.parameters()):
            new_weight = tau*local_param.data + (1.0-tau)*target_param.data

            target_param.data.copy_(new_weight)

    def learn(self,experiences):
        loss = self.compute_loss(experiences)
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

        self.soft_update()
    
    def get_action(self,state):
        state_t = torch.from_numpy(state).float().unsqueeze(0)
        state_t = state_t.to(self.device)


        with torch.no_grad():
            q_values = self.q_network(state_t)

        self.q_network.train()

        if random.random() < self.epsilon:
            return random.randrange(self.n_actions)
        else:
            return torch.argmax(q_values[0]).item()
        
    def train(self,n_episodes=4000,n_steps=1000,mini_batch_size=64):
        memory_buffer = deque(maxlen=self.memory_size)
        self.target_q_network.load_state_dict(self.q_network.state_dict())
        
        for i in tqdm(range(n_episodes)):
            state , info = self.env.reset()

            for t in range(n_steps):
                action = self.get_action(state)

                next_state , reward , terminated, truncated , next_info = self.env.step(action)

                done = terminated or truncated
                memory_buffer.append(self.experience(state, action, reward, next_state, done))

                if len(memory_buffer) > mini_batch_size and t% self.num_steps_for_update==0:

                    samples = random.sample(memory_buffer, k=mini_batch_size)

                    states = torch.from_numpy(np.vstack([e.state for e in samples if e is not None])).float()
                    actions = torch.from_numpy(np.vstack([e.action for e in samples if e is not None])).float()
                    rewards = torch.from_numpy(np.vstack([e.reward for e in samples if e is not None])).float()
                    next_states = torch.from_numpy(np.vstack([e.next_state for e in samples if e is not None])).float()
                    dones = torch.from_numpy(np.vstack([e.done for e in samples if e is not None]).astype(np.uint8)).float()
                    experiences_tensor = (states, actions.squeeze(), rewards.squeeze(), next_states, dones.squeeze())

                    self.learn(experiences_tensor)
                state = next_state.copy()

                if done:
                    break
            self.epsilon = max(0.01,0.995*self.epsilon)
    
    def save_agent(self,name:str):
        torch.save(self.q_network.state_dict(), name)

    def load_agent(self,name):
        self.q_network.load_state_dict(torch.load(name))
        self.q_network.eval()
        self.epsilon = 0.01



In [3]:

gamma = 0.995
learning_rate = 0.001
num_steps_for_update = 4
epsilon = 1
n_episodes = 4000
n_steps = 1000
tau = 0.001
mini_badge_size = 64
agent = lunar_agent(gamma=gamma,num_steps_for_update=num_steps_for_update,learning_rate=learning_rate,epsilon=epsilon)

In [4]:
agent.create_nn()

In [26]:
agent.train(n_episodes,n_steps,mini_badge_size)

100%|██████████| 4000/4000 [30:55<00:00,  2.16it/s] 


In [6]:
env = gym.make("LunarLander-v3",render_mode = "human")

for i in range(10):
    state , info = env.reset()
    done = False

    while not done:
        with torch.no_grad():

            action = agent.get_action(state)
            state , reward, terminated, truncated , next_info = env.step(action)
            done = terminated or truncated
env.close()

In [28]:
agent.save_agent("lunar_agent")

In [5]:
agent.load_agent("lunar_agent")

C:\Users\User\AppData\Local\Temp\ipykernel_8884\1547095692.py:136: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.q_network.load_state_dict(torch.load(name))
